# 03 — Documentação do treinamento

Este notebook documenta o plano de treinamento dos cenários experimentais com ajuste do modelo.

A etapa anterior, `02_dataset_creation.ipynb`, construiu a base canônica, os exemplos atacados e as views específicas para cada mecanismo de defesa. Este notebook parte desses artefatos e organiza a documentação necessária antes da execução dos treinamentos.

O objetivo aqui não é ainda executar o fine-tuning pesado. Em vez disso, este notebook registra:

```text
- quais cenários exigem treinamento;
- quais arquivos serão usados por cada cenário;
- qual modelo base será usado;
- quais adaptadores serão produzidos;
- quais configurações de LoRA/QLoRA, SFT e DPO serão adotadas;
- quais arquivos precisam existir antes do treinamento;
- quais riscos e limitações precisam ser documentados;
- quais manifestos serão usados pelos notebooks seguintes.
```

Essa separação ajuda a manter o experimento organizado e reprodutível. Antes de treinar C2, C3 e C4, é importante garantir que todos os arquivos de entrada existem, que os tamanhos estão corretos e que o plano de treinamento está explícito.

## 1. Objetivo do notebook

Este notebook tem como objetivo documentar e validar o plano de treinamento dos modelos defendidos.

No desenho experimental deste projeto, os cenários são:

```text
C0 — Base model
C1 — StruQ format-only
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

Os cenários C0 e C1 não exigem treinamento. O C0 corresponde ao modelo base avaliado diretamente. O C1 usa o mesmo modelo base, mas altera a forma de apresentar a entrada, aplicando uma estrutura inspirada em StruQ.

Já os cenários C2, C3 e C4 exigem ajuste do modelo. Esses cenários partem do mesmo modelo base e produzem adaptadores diferentes:

```text
C2 — adaptador treinado com StruQ-like SFT
C3 — adaptador treinado com SecAlign-like DPO
C4 — adaptador treinado com Instruction-Hierarchy-like SFT
```

A função deste notebook é preparar a documentação e os arquivos de configuração para esses três treinamentos. Isso inclui conferir os arquivos de entrada, registrar os diretórios de saída, definir hiperparâmetros iniciais e criar um manifesto do plano de treinamento.

Ao final, este notebook deve produzir:

```text
configs/training_plan.yaml

manifests/training/03_training_documentation_manifest.json
manifests/training/03_training_documentation_manifest.md
```

O arquivo `training_plan.yaml` será usado como referência pelos notebooks ou scripts que executarão o treinamento de fato.

## 2. Imports e parâmetros globais

Nesta etapa, serão importadas as bibliotecas necessárias para validar os artefatos do dataset, carregar os manifestos anteriores e gerar a documentação do treinamento.

Este notebook segue a mesma convenção dos notebooks anteriores:

```text
PROJECT_ROOT = /workspace/pi-defense-exp
EXPECTED_KERNEL = Python (pi-defense-exp)
EXPECTED_PYTHON = /workspace/pi-defense-exp/.venv/bin/python
```

A validação do Python atual é importante porque este notebook deve ser executado no mesmo ambiente virtual configurado no notebook `01_environment_setup.ipynb`.

Além disso, este notebook usa diretórios já previstos na estrutura do projeto:

```text
configs/
data/views/
adapters/
logs/training/
manifests/training/
results/
```

A pasta `adapters/` será usada para armazenar os adaptadores LoRA/QLoRA produzidos pelos treinamentos. A pasta `manifests/training/` armazenará o manifesto gerado por este notebook.

In [1]:
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path("/workspace/pi-defense-exp")
EXPECTED_KERNEL = "Python (pi-defense-exp)"
EXPECTED_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
CURRENT_PYTHON = Path(sys.executable)

CONFIG_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
VIEWS_DIR = DATA_DIR / "views"
CANONICAL_DIR = DATA_DIR / "canonical"

ADAPTERS_DIR = PROJECT_ROOT / "adapters"
RESULTS_DIR = PROJECT_ROOT / "results"
LOG_DIR = PROJECT_ROOT / "logs" / "training"
MANIFEST_DIR = PROJECT_ROOT / "manifests" / "training"

TRAINING_PLAN_PATH = CONFIG_DIR / "training_plan.yaml"
DATASET_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "data" / "02_dataset_creation_manifest.json"
ENVIRONMENT_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "environment" / "01_environment_setup_manifest.json"
EXPERIMENT_CONFIG_PATH = CONFIG_DIR / "experiment.yaml"

for path in [
    CONFIG_DIR,
    ADAPTERS_DIR,
    ADAPTERS_DIR / "struq",
    ADAPTERS_DIR / "secalign",
    ADAPTERS_DIR / "ih",
    LOG_DIR,
    MANIFEST_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python atual:", CURRENT_PYTHON)
print("Python esperado:", EXPECTED_PYTHON)
print("Training manifest dir:", MANIFEST_DIR)

Project root: /workspace/pi-defense-exp
Python atual: /workspace/pi-defense-exp/.venv/bin/python
Python esperado: /workspace/pi-defense-exp/.venv/bin/python
Training manifest dir: /workspace/pi-defense-exp/manifests/training


## 3. Validar kernel e ambiente

Antes de documentar ou validar qualquer arquivo de treinamento, é necessário confirmar que este notebook está sendo executado no Python correto.

A validação usa o caminho esperado do interpretador Python dentro da `.venv`:

```text
/workspace/pi-defense-exp/.venv/bin/python
```

Se o notebook estiver usando outro kernel, a execução será interrompida. Isso evita que o plano de treinamento seja gerado em um ambiente diferente daquele usado para preparar os dados.

In [2]:
print("Python atual:", CURRENT_PYTHON)
print("Python esperado para este notebook:", EXPECTED_PYTHON)

if CURRENT_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        "Este notebook deve ser executado com o kernel 'Python (pi-defense-exp)'. "
        "Selecione esse kernel no Jupyter antes de continuar."
    )

print("Validação inicial concluída.")
print("Kernel correto para o notebook 03.")

Python atual: /workspace/pi-defense-exp/.venv/bin/python
Python esperado para este notebook: /workspace/pi-defense-exp/.venv/bin/python
Validação inicial concluída.
Kernel correto para o notebook 03.


## 4. Carregar configurações e manifestos anteriores

Nesta etapa, serão carregados os arquivos de configuração e os manifestos produzidos pelos notebooks anteriores.

O arquivo `configs/experiment.yaml` contém parâmetros gerais do experimento, como seed, tasks, tamanhos dos splits e tipos de ataque. Ele foi criado no notebook de configuração do ambiente.

O manifesto do dataset, `manifests/data/02_dataset_creation_manifest.json`, registra os arquivos gerados pelo notebook 02, incluindo dados canônicos, views de treinamento e arquivos comuns de avaliação.

Este notebook usa esses arquivos para garantir que o plano de treinamento seja compatível com os dados realmente produzidos. Se o manifesto do dataset não existir, a execução deve ser interrompida, porque não é seguro documentar o treinamento sem confirmar quais arquivos foram criados.

In [3]:
def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_yaml(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo de configuração não encontrado: {EXPERIMENT_CONFIG_PATH}"
    )

if not DATASET_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifesto do dataset não encontrado: {DATASET_MANIFEST_PATH}. "
        "Execute o notebook 02 antes de continuar."
    )

experiment_config = load_yaml(EXPERIMENT_CONFIG_PATH)
dataset_manifest = load_json(DATASET_MANIFEST_PATH)

environment_manifest = None
if ENVIRONMENT_MANIFEST_PATH.exists():
    environment_manifest = load_json(ENVIRONMENT_MANIFEST_PATH)

print("Configuração do experimento carregada de:", EXPERIMENT_CONFIG_PATH)
print("Manifesto do dataset carregado de:", DATASET_MANIFEST_PATH)

if environment_manifest is not None:
    print("Manifesto do ambiente carregado de:", ENVIRONMENT_MANIFEST_PATH)
else:
    print("Manifesto do ambiente não encontrado. O notebook continuará, mas isso será registrado no manifesto final.")

Configuração do experimento carregada de: /workspace/pi-defense-exp/configs/experiment.yaml
Manifesto do dataset carregado de: /workspace/pi-defense-exp/manifests/data/02_dataset_creation_manifest.json
Manifesto do ambiente carregado de: /workspace/pi-defense-exp/manifests/environment/01_environment_setup_manifest.json


## 5. Revisão dos cenários experimentais

Nesta etapa, os cenários experimentais são revisados antes do treinamento.

A comparação final envolve cinco cenários:

| Cenário | Nome | Treinamento? | Função |
|---|---|---:|---|
| C0 | Base model | Não | Baseline sem defesa |
| C1 | StruQ format-only | Não | Avalia apenas a formatação estruturada |
| C2 | StruQ-like SFT | Sim | Avalia ajuste supervisionado com separação entre instrução e dado |
| C3 | SecAlign-like DPO | Sim | Avalia otimização por preferência usando pares chosen/rejected |
| C4 | Instruction-Hierarchy-like SFT | Sim | Avalia ajuste supervisionado com níveis de autoridade |

A regra metodológica mais importante é que todos os cenários devem partir do mesmo modelo base e ser avaliados nos mesmos arquivos comuns de teste.

Assim, as diferenças observadas na avaliação final devem estar associadas ao mecanismo de defesa ou à forma de apresentação da entrada, e não a mudanças nos exemplos de teste.

In [4]:
scenario_rows = [
    {
        "scenario": "C0",
        "name": "Base model",
        "requires_training": False,
        "training_method": "none",
        "adapter_output": None,
        "role": "Baseline sem defesa.",
    },
    {
        "scenario": "C1",
        "name": "StruQ format-only",
        "requires_training": False,
        "training_method": "none",
        "adapter_output": None,
        "role": "Modelo base avaliado com prompt estruturado, sem fine-tuning.",
    },
    {
        "scenario": "C2",
        "name": "StruQ-like SFT",
        "requires_training": True,
        "training_method": "SFT",
        "adapter_output": str(ADAPTERS_DIR / "struq"),
        "role": "Treinamento supervisionado com instrução confiável e dado não confiável separados.",
    },
    {
        "scenario": "C3",
        "name": "SecAlign-like DPO",
        "requires_training": True,
        "training_method": "DPO",
        "adapter_output": str(ADAPTERS_DIR / "secalign"),
        "role": "Otimização por preferência com chosen=expected_answer e rejected=attack_target.",
    },
    {
        "scenario": "C4",
        "name": "Instruction-Hierarchy-like SFT",
        "requires_training": True,
        "training_method": "SFT",
        "adapter_output": str(ADAPTERS_DIR / "ih"),
        "role": "Treinamento supervisionado com papéis system, user e tool.",
    },
]

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

,scenario,name,requires_training,training_method,adapter_output,role
0,C0,Base model,False,none,NaN,Baseline sem defesa.
1,C1,StruQ format-only,False,none,NaN,"Modelo base avaliado com prompt estruturado, s..."
2,C2,StruQ-like SFT,True,SFT,/workspace/pi-defense-exp/adapters/struq,Treinamento supervisionado com instrução confi...
3,C3,SecAlign-like DPO,True,DPO,/workspace/pi-defense-exp/adapters/secalign,Otimização por preferência com chosen=expected...
4,C4,Instruction-Hierarchy-like SFT,True,SFT,/workspace/pi-defense-exp/adapters/ih,"Treinamento supervisionado com papéis system, ..."


## 6. Arquivos de treino e validação por cenário

Nesta etapa, são definidos os arquivos de entrada usados por cada cenário com treinamento.

As views foram criadas no notebook `02_dataset_creation.ipynb`. Cada view reorganiza a mesma base canônica em um formato adequado para uma estratégia de defesa.

Para evitar ambiguidade entre o rótulo conceitual do cenário e o identificador usado nos caminhos, este notebook passa a usar dois nomes:

```text
scenario_label = C2, C3, C4
scenario_id    = c2_struq_sft, c3_secalign_dpo, c4_ih_sft
```

O `scenario_label` é o nome curto usado na descrição metodológica. Já o `scenario_id` é o identificador operacional usado em arquivos, diretórios, logs, manifestos e no notebook `04_run_training.ipynb`.

Essa padronização é importante porque os logs de treinamento são separados por cenário e seed. Portanto, o plano documentado no notebook 03 deve usar os mesmos `scenario_id` que serão usados pelo notebook 04.

Para SFT, usamos exemplos limpos e exemplos atacados vistos:

```text
train_clean + train_attacked_seen
validation_clean + validation_attacked_seen
```

Isso vale para:

```text
C2 / c2_struq_sft — StruQ-like SFT
C4 / c4_ih_sft — Instruction-Hierarchy-like SFT
```

Para DPO, usamos apenas exemplos atacados vistos, porque o treinamento por preferência exige um par:

```text
chosen = expected_answer
rejected = attack_target
```

Como exemplos limpos não possuem `attack_target`, eles não entram diretamente na view SecAlign-like DPO.

Os arquivos esperados são:

| Scenario label | Scenario ID | Método | Train | Validation |
|---|---|---|---|---|
| C2 | `c2_struq_sft` | SFT | `data/views/struq/train_sft.jsonl` | `data/views/struq/validation_sft.jsonl` |
| C3 | `c3_secalign_dpo` | DPO | `data/views/secalign/train_dpo.jsonl` | `data/views/secalign/validation_dpo.jsonl` |
| C4 | `c4_ih_sft` | SFT | `data/views/ih/train_sft.jsonl` | `data/views/ih/validation_sft.jsonl` |


In [5]:
training_files = {
    "c2_struq_sft": {
        "scenario_label": "C2",
        "name": "StruQ-like SFT",
        "method": "sft",
        "train": VIEWS_DIR / "struq" / "train_sft.jsonl",
        "validation": VIEWS_DIR / "struq" / "validation_sft.jsonl",
        "adapter_output": ADAPTERS_DIR / "struq",
    },
    "c3_secalign_dpo": {
        "scenario_label": "C3",
        "name": "SecAlign-like DPO",
        "method": "dpo",
        "train": VIEWS_DIR / "secalign" / "train_dpo.jsonl",
        "validation": VIEWS_DIR / "secalign" / "validation_dpo.jsonl",
        "adapter_output": ADAPTERS_DIR / "secalign",
    },
    "c4_ih_sft": {
        "scenario_label": "C4",
        "name": "Instruction-Hierarchy-like SFT",
        "method": "sft",
        "train": VIEWS_DIR / "ih" / "train_sft.jsonl",
        "validation": VIEWS_DIR / "ih" / "validation_sft.jsonl",
        "adapter_output": ADAPTERS_DIR / "ih",
    },
}


def count_jsonl_lines(path: Path) -> int:
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)


training_file_rows = []

for scenario_id, info in training_files.items():
    for split_name in ["train", "validation"]:
        path = info[split_name]
        training_file_rows.append({
            "scenario_id": scenario_id,
            "scenario_label": info["scenario_label"],
            "name": info["name"],
            "method": info["method"],
            "split": split_name,
            "path": str(path),
            "exists": path.exists(),
            "line_count": count_jsonl_lines(path) if path.exists() else None,
        })

training_files_df = pd.DataFrame(training_file_rows)
display(training_files_df)

missing_files = training_files_df.loc[~training_files_df["exists"], "path"].tolist()
if missing_files:
    raise FileNotFoundError("Arquivos de treinamento ausentes: " + ", ".join(missing_files))


,scenario_id,scenario_label,name,method,split,path,exists,line_count
0,c2_struq_sft,C2,StruQ-like SFT,sft,train,/workspace/pi-defense-exp/data/views/struq/tra...,True,4200
1,c2_struq_sft,C2,StruQ-like SFT,sft,validation,/workspace/pi-defense-exp/data/views/struq/val...,True,700
2,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,train,/workspace/pi-defense-exp/data/views/secalign/...,True,2100
3,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,validation,/workspace/pi-defense-exp/data/views/secalign/...,True,350
4,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,train,/workspace/pi-defense-exp/data/views/ih/train_...,True,4200
5,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,validation,/workspace/pi-defense-exp/data/views/ih/valida...,True,700


## 7. Validar contagens das views de treinamento

Nesta etapa, as contagens das views são conferidas antes de documentar o plano de treinamento.

As contagens esperadas vêm diretamente do desenho do dataset:

| View | Train | Validation |
|---|---:|---:|
| StruQ-like SFT | 4.200 | 700 |
| SecAlign-like DPO | 2.100 | 350 |
| Instruction-Hierarchy-like SFT | 4.200 | 700 |

A diferença entre SFT e DPO é esperada. As views de SFT usam exemplos limpos e atacados. Já a view DPO usa apenas exemplos atacados, porque precisa do campo `attack_target` para construir o par `chosen` e `rejected`.

Se alguma contagem estiver incorreta, o treinamento não deve ser iniciado, pois isso indicaria erro na criação dos dados ou no caminho dos arquivos.

In [6]:
expected_training_counts = {
    ("c2_struq_sft", "train"): 4200,
    ("c2_struq_sft", "validation"): 700,
    ("c3_secalign_dpo", "train"): 2100,
    ("c3_secalign_dpo", "validation"): 350,
    ("c4_ih_sft", "train"): 4200,
    ("c4_ih_sft", "validation"): 700,
}

for _, row in training_files_df.iterrows():
    key = (row["scenario_id"], row["split"])
    expected = expected_training_counts[key]
    observed = row["line_count"]

    if observed != expected:
        raise RuntimeError(
            f"Contagem inesperada para {key}: esperado={expected}, observado={observed}."
        )

print("Contagens das views de treinamento validadas com sucesso.")


Contagens das views de treinamento validadas com sucesso.


## 8. Modelo base

Nesta etapa, será definido o modelo base usado pelos cenários experimentais.

A decisão metodológica central é que todos os cenários devem partir do mesmo checkpoint base. Isso reduz fatores de confusão e torna a comparação mais controlada.

O modelo base inicial sugerido para este experimento é:

```text
Qwen/Qwen2.5-1.5B-Instruct
```

Esse modelo é pequeno o suficiente para ser mais viável em um ambiente RunPod com uma única GPU, mas ainda é um modelo instrucional capaz de seguir prompts e responder tarefas de classificação textual.

A escolha do modelo pode ser alterada antes da execução do treinamento, mas deve permanecer igual para C0, C1, C2, C3 e C4.

A regra é:

```text
Mesmo modelo base para todos os cenários.
Adaptadores diferentes apenas para C2, C3 e C4.
Mesmos arquivos de avaliação para todos os cenários.
```

Caso o ambiente suporte um modelo maior, como Llama 3.1 8B Instruct, essa troca pode ser feita posteriormente. No entanto, para manter a viabilidade computacional, este notebook documenta uma configuração inicial mais leve.

In [7]:
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

model_plan = {
    "base_model_id": BASE_MODEL_ID,
    "model_family": "causal_lm",
    "uses_chat_template": True,
    "reason_for_choice": (
        "Modelo instrucional relativamente pequeno, mais viável para uma GPU no RunPod "
        "e adequado para avaliação inicial de prompts e adaptadores."
    ),
    "same_base_model_for_all_scenarios": True,
}

model_plan

{'base_model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
 'model_family': 'causal_lm',
 'uses_chat_template': True,
 'reason_for_choice': 'Modelo instrucional relativamente pequeno, mais viável para uma GPU no RunPod e adequado para avaliação inicial de prompts e adaptadores.',
 'same_base_model_for_all_scenarios': True}

## 8.1. Seeds experimentais

Nesta etapa, são registradas as seeds experimentais que serão usadas nos notebooks de treinamento e avaliação.

É importante separar dois conceitos diferentes:

```text
DATASET_SEED = 42
EXPERIMENT_SEEDS = [42, 123, 2026]
```

A `DATASET_SEED` representa a seed usada na criação da base experimental. Ela controla a amostragem, os splits e a geração determinística dos exemplos atacados no notebook `02_dataset_creation.ipynb`. Essa seed deve permanecer fixa para que todos os cenários usem exatamente os mesmos dados de treino, validação e teste.

Já `EXPERIMENT_SEEDS` representa as réplicas de treinamento. Essas seeds serão usadas no notebook `04_run_training.ipynb` para treinar os adaptadores de C2, C3 e C4 mais de uma vez, reduzindo a dependência dos resultados em uma única inicialização, uma única ordem de batches ou uma única trajetória de otimização.

Neste notebook, o rótulo metodológico do cenário continua sendo `C2`, `C3` e `C4`. Porém, para diretórios, logs e manifestos, os identificadores operacionais devem ser exatamente os mesmos usados no notebook 04:

```text
c2_struq_sft
c3_secalign_dpo
c4_ih_sft
```

Assim, os cenários com treinamento serão executados uma vez para cada seed:

```text
C2 / c2_struq_sft — StruQ-like SFT: seeds 42, 123, 2026
C3 / c3_secalign_dpo — SecAlign-like DPO: seeds 42, 123, 2026
C4 / c4_ih_sft — Instruction-Hierarchy-like SFT: seeds 42, 123, 2026
```

Isso produz nove treinamentos de adaptadores no total:

```text
3 cenários treináveis × 3 seeds = 9 execuções de treinamento
```

Os cenários C0 e C1 não treinam adaptadores. Eles usam o modelo base diretamente, com ou sem formatação defensiva. Por isso, as seeds experimentais são relevantes principalmente para C2, C3 e C4. Na avaliação, C0 e C1 só precisam de múltiplas seeds se a geração for estocástica. Caso a avaliação seja feita com `temperature=0`, esses cenários podem ser avaliados uma única vez.

Cada adaptador treinado deve ser salvo em um diretório que inclua a seed. Isso evita sobrescrita entre réplicas e permite associar cada resultado ao treinamento que o produziu.

Exemplo esperado de organização dos adaptadores:

```text
adapters/struq/seed_42/
adapters/struq/seed_123/
adapters/struq/seed_2026/

adapters/secalign/seed_42/
adapters/secalign/seed_123/
adapters/secalign/seed_2026/

adapters/ih/seed_42/
adapters/ih/seed_123/
adapters/ih/seed_2026/
```

O manifesto deste notebook também deve registrar essas seeds e os `scenario_id`, porque eles fazem parte do plano metodológico do treinamento. Os notebooks posteriores devem usar essa informação para executar as réplicas, salvar os adaptadores, registrar logs e calcular médias e desvios padrão das métricas finais.


In [8]:
DATASET_SEED = int(dataset_manifest.get("seed", 42))
EXPERIMENT_SEEDS = [42, 123, 2026]
TRAINED_SCENARIOS = ["c2_struq_sft", "c3_secalign_dpo", "c4_ih_sft"]
TRAINED_SCENARIO_LABELS = ["C2", "C3", "C4"]
REPLICATES_PER_TRAINED_SCENARIO = len(EXPERIMENT_SEEDS)
TOTAL_ADAPTER_RUNS = len(TRAINED_SCENARIOS) * REPLICATES_PER_TRAINED_SCENARIO

seed_plan = {
    "dataset_seed": DATASET_SEED,
    "experiment_seeds": EXPERIMENT_SEEDS,
    "trained_scenario_ids": TRAINED_SCENARIOS,
    "trained_scenario_labels": TRAINED_SCENARIO_LABELS,
    "replicates_per_trained_scenario": REPLICATES_PER_TRAINED_SCENARIO,
    "total_adapter_runs": TOTAL_ADAPTER_RUNS,
    "notes": [
        "DATASET_SEED controls dataset construction and must stay fixed across scenarios.",
        "EXPERIMENT_SEEDS define training replicates for c2_struq_sft, c3_secalign_dpo, and c4_ih_sft.",
        "C0 and C1 do not train adapters.",
        "Scenario IDs are kept aligned with 04_run_training.ipynb.",
    ],
}

seed_plan


{'dataset_seed': 42,
 'experiment_seeds': [42, 123, 2026],
 'trained_scenario_ids': ['c2_struq_sft', 'c3_secalign_dpo', 'c4_ih_sft'],
 'trained_scenario_labels': ['C2', 'C3', 'C4'],
 'replicates_per_trained_scenario': 3,
 'total_adapter_runs': 9,
 'notes': ['DATASET_SEED controls dataset construction and must stay fixed across scenarios.',
  'EXPERIMENT_SEEDS define training replicates for c2_struq_sft, c3_secalign_dpo, and c4_ih_sft.',
  'C0 and C1 do not train adapters.',
  'Scenario IDs are kept aligned with 04_run_training.ipynb.']}

## 9. Estratégia de treinamento para C2 — StruQ-like SFT

O cenário C2 usa treinamento supervisionado em uma view estruturada no estilo StruQ.

A view separa explicitamente:

```text
[TRUSTED_INSTRUCTION]
...
[/TRUSTED_INSTRUCTION]

[UNTRUSTED_DATA]
...
[/UNTRUSTED_DATA]
```

O objetivo do treinamento é ensinar o modelo a seguir a instrução confiável e tratar o conteúdo do bloco não confiável apenas como dado. Mesmo quando esse bloco contém uma injeção, a resposta esperada continua sendo o rótulo correto da tarefa original.

A view usada será:

```text
data/views/struq/train_sft.jsonl
data/views/struq/validation_sft.jsonl
```

Cada exemplo possui:

```text
prompt
completion
task_name
attack_type
source_id
label_space
```

O adaptador treinado deve ser salvo em:

```text
adapters/struq/
```

Esse adaptador será usado posteriormente na avaliação do cenário C2.

## 10. Estratégia de treinamento para C3 — SecAlign-like DPO

O cenário C3 usa otimização por preferência em uma view SecAlign-like.

Nesse formato, cada exemplo atacado é transformado em um par de preferência:

```text
chosen = expected_answer
rejected = attack_target
```

O campo `chosen` representa a resposta segura e correta, isto é, a resposta que segue a instrução legítima. O campo `rejected` representa a resposta indesejada, isto é, o alvo da injeção.

A view usada será:

```text
data/views/secalign/train_dpo.jsonl
data/views/secalign/validation_dpo.jsonl
```

Cada exemplo possui:

```text
prompt
chosen
rejected
task_name
attack_type
source_id
label_space
```

O adaptador treinado deve ser salvo em:

```text
adapters/secalign/
```

Essa implementação é uma aproximação experimental de SecAlign. Ela não reproduz integralmente o pipeline original, mas preserva o princípio central: preferir respostas que seguem a tarefa legítima e rejeitar respostas que seguem a injeção.

## 11. Estratégia de treinamento para C4 — Instruction-Hierarchy-like SFT

O cenário C4 usa treinamento supervisionado com uma estrutura de mensagens inspirada em Instruction Hierarchy.

A view explicita níveis de autoridade usando os papéis:

```text
system
user
tool
```

A mensagem `system` define uma regra geral de segurança: dados externos podem conter instruções não confiáveis e não devem sobrescrever a tarefa do usuário.

A mensagem `user` contém a instrução legítima da tarefa.

A mensagem `tool` contém o dado externo. Em exemplos limpos, esse dado é apenas a entrada original da tarefa. Em exemplos atacados, esse dado contém a entrada original acompanhada da instrução adversarial.

A view usada será:

```text
data/views/ih/train_sft.jsonl
data/views/ih/validation_sft.jsonl
```

Cada exemplo possui:

```text
messages
completion
task_name
attack_type
source_id
label_space
```

O adaptador treinado deve ser salvo em:

```text
adapters/ih/
```

Essa view não pretende reproduzir integralmente um dataset oficial de hierarquia de instruções. Ela implementa uma versão controlada do princípio de hierarquia: instruções de maior autoridade devem prevalecer sobre comandos inseridos em dados externos.

## 12. Configuração LoRA/QLoRA

Nesta etapa, serão documentadas as configurações iniciais de LoRA/QLoRA.

Como o experimento será executado em uma GPU no RunPod, a estratégia recomendada é usar adaptação eficiente em parâmetros, em vez de fine-tuning completo do modelo.

A configuração inicial sugerida usa LoRA com quantização em 4 bits quando compatível com o ambiente. Essa estratégia reduz o uso de memória e torna mais viável treinar adaptadores separados para C2, C3 e C4.

Configuração inicial sugerida:

```text
load_in_4bit = true
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05
target_modules = q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
```

Esses valores podem ser ajustados depois de testes iniciais de memória e estabilidade. O mais importante é manter configurações equivalentes entre C2, C3 e C4 sempre que possível, para preservar a comparabilidade.

Caso o modelo escolhido não use exatamente esses nomes de módulos, os notebooks de treinamento deverão verificar os nomes disponíveis no modelo antes de aplicar LoRA.

In [9]:
lora_config = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_use_double_quant": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
}

lora_config

{'load_in_4bit': True,
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_compute_dtype': 'bfloat16',
 'bnb_4bit_use_double_quant': True,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'bias': 'none',
 'task_type': 'CAUSAL_LM',
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj']}

## 13. Configurações de SFT

Os cenários C2 e C4 usam SFT.

C2 treina com a view StruQ-like:

```text
data/views/struq/train_sft.jsonl
```

C4 treina com a view Instruction-Hierarchy-like:

```text
data/views/ih/train_sft.jsonl
```

A configuração inicial deve ser pequena e conservadora, porque o objetivo do trabalho é uma avaliação experimental de disciplina, não maximizar desempenho em larga escala.

Configuração inicial sugerida:

```text
epochs = 1
learning_rate = 2e-4
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
max_seq_length = 1024
logging_steps = 10
save_strategy = epoch
evaluation_strategy = epoch
```

Esses valores podem ser ajustados de acordo com a memória disponível. Se houver erro de memória, as primeiras opções de redução são:

```text
- diminuir max_seq_length;
- aumentar gradient_accumulation_steps mantendo batch size 1;
- usar modelo base menor;
- reduzir número de exemplos em um teste piloto.
```

In [10]:
sft_training_config = {
    "num_train_epochs": 1,
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_seq_length": 1024,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "fp16": False,
    "bf16": True,
}

sft_training_config

{'num_train_epochs': 1,
 'learning_rate': 0.0002,
 'per_device_train_batch_size': 1,
 'per_device_eval_batch_size': 1,
 'gradient_accumulation_steps': 8,
 'max_seq_length': 1024,
 'logging_steps': 10,
 'save_strategy': 'epoch',
 'eval_strategy': 'epoch',
 'warmup_ratio': 0.03,
 'lr_scheduler_type': 'cosine',
 'fp16': False,
 'bf16': True}

## 14. Configurações de DPO

O cenário C3 usa DPO.

A view SecAlign-like DPO contém pares `chosen` e `rejected`, construídos a partir de exemplos atacados vistos.

Configuração inicial sugerida:

```text
epochs = 1
learning_rate = 5e-5
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
max_prompt_length = 1024
max_length = 1280
beta = 0.1
```

O valor de `beta` controla a força da preferência no DPO. Um valor inicial comum é `0.1`, mas ele pode ser ajustado em experimentos posteriores.

Como DPO costuma ser mais sensível e consumir mais memória que SFT, pode ser necessário reduzir `max_prompt_length`, `max_length` ou a quantidade de exemplos durante um teste piloto.

In [11]:
dpo_training_config = {
    "num_train_epochs": 1,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_prompt_length": 1024,
    "max_length": 1280,
    "beta": 0.1,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "fp16": False,
    "bf16": True,
}

dpo_training_config

{'num_train_epochs': 1,
 'learning_rate': 5e-05,
 'per_device_train_batch_size': 1,
 'per_device_eval_batch_size': 1,
 'gradient_accumulation_steps': 8,
 'max_prompt_length': 1024,
 'max_length': 1280,
 'beta': 0.1,
 'logging_steps': 10,
 'save_strategy': 'epoch',
 'eval_strategy': 'epoch',
 'warmup_ratio': 0.03,
 'lr_scheduler_type': 'cosine',
 'fp16': False,
 'bf16': True}

## 15. Diretórios esperados para adaptadores e logs

Nesta etapa, são definidos os diretórios de saída dos treinamentos.

Como o experimento agora possui múltiplas seeds experimentais, cada cenário treinável deve salvar um adaptador separado para cada seed. Isso é necessário para evitar sobrescrita de resultados e para permitir que a avaliação posterior associe cada métrica à réplica correta.

A organização esperada passa a ser:

```text
adapters/struq/seed_42/
adapters/struq/seed_123/
adapters/struq/seed_2026/

adapters/secalign/seed_42/
adapters/secalign/seed_123/
adapters/secalign/seed_2026/

adapters/ih/seed_42/
adapters/ih/seed_123/
adapters/ih/seed_2026/
```

A pasta do cenário continua funcionando como diretório-base do método defensivo:

```text
adapters/struq/
adapters/secalign/
adapters/ih/
```

Mas os artefatos finais de treinamento devem ser gravados dentro das subpastas `seed_<valor>`.

Os logs de treinamento devem ser salvos em:

```text
logs/training/
```

Os resultados intermediários, checkpoints ou relatórios produzidos pelos scripts de treinamento podem ser salvos em subpastas específicas, desde que o manifesto do treinamento registre os caminhos usados.

A regra geral é:

```text
Um cenário, uma seed, um diretório de saída.
```

Essa organização é importante porque C2, C3 e C4 serão treinados três vezes cada um, usando as seeds `42`, `123` e `2026`. Assim, o experimento passa a ter nove execuções de treinamento de adaptadores.


In [12]:
adapter_rows = []

for scenario_id, info in training_files.items():
    scenario_base_dir = info["adapter_output"]
    scenario_base_dir.mkdir(parents=True, exist_ok=True)

    for seed in EXPERIMENT_SEEDS:
        output_dir = scenario_base_dir / f"seed_{seed}"
        output_dir.mkdir(parents=True, exist_ok=True)

        adapter_rows.append({
            "scenario_id": scenario_id,
            "scenario_label": info["scenario_label"],
            "name": info["name"],
            "method": info["method"],
            "seed": seed,
            "adapter_output_base": str(scenario_base_dir),
            "adapter_output": str(output_dir),
            "exists": output_dir.exists(),
        })

adapter_dirs_df = pd.DataFrame(adapter_rows)
display(adapter_dirs_df)


,scenario_id,scenario_label,name,method,seed,adapter_output_base,adapter_output,exists
0,c2_struq_sft,C2,StruQ-like SFT,sft,42,/workspace/pi-defense-exp/adapters/struq,/workspace/pi-defense-exp/adapters/struq/seed_42,True
1,c2_struq_sft,C2,StruQ-like SFT,sft,123,/workspace/pi-defense-exp/adapters/struq,/workspace/pi-defense-exp/adapters/struq/seed_123,True
2,c2_struq_sft,C2,StruQ-like SFT,sft,2026,/workspace/pi-defense-exp/adapters/struq,/workspace/pi-defense-exp/adapters/struq/seed_...,True
3,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,42,/workspace/pi-defense-exp/adapters/secalign,/workspace/pi-defense-exp/adapters/secalign/se...,True
4,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,123,/workspace/pi-defense-exp/adapters/secalign,/workspace/pi-defense-exp/adapters/secalign/se...,True
5,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,2026,/workspace/pi-defense-exp/adapters/secalign,/workspace/pi-defense-exp/adapters/secalign/se...,True
6,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,42,/workspace/pi-defense-exp/adapters/ih,/workspace/pi-defense-exp/adapters/ih/seed_42,True
7,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,123,/workspace/pi-defense-exp/adapters/ih,/workspace/pi-defense-exp/adapters/ih/seed_123,True
8,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,2026,/workspace/pi-defense-exp/adapters/ih,/workspace/pi-defense-exp/adapters/ih/seed_2026,True


## 15.1. Plano de logs incrementais

Nesta etapa, é documentada a estratégia de logs que será usada no notebook `04_run_training.ipynb`.

Essa documentação é importante porque o treinamento não será uma única execução simples. O experimento possui três cenários treináveis:

```text
C2 / c2_struq_sft — StruQ-like SFT
C3 / c3_secalign_dpo — SecAlign-like DPO
C4 / c4_ih_sft — Instruction-Hierarchy-like SFT
```

Os nomes `C2`, `C3` e `C4` são rótulos metodológicos. Para logs, diretórios e manifestos, o identificador operacional deve ser o `scenario_id` usado pelo notebook 04:

```text
c2_struq_sft
c3_secalign_dpo
c4_ih_sft
```

Cada um desses cenários será executado com três seeds experimentais:

```text
42, 123, 2026
```

Portanto, o treinamento completo envolve nove runs de adaptadores. Em uma execução longa, é possível que algumas runs sejam concluídas e outras falhem por falta de memória, interrupção do ambiente, problema de autenticação, erro de configuração ou instabilidade da GPU.

Por isso, o notebook de treinamento deve salvar logs de forma incremental. Isso significa registrar eventos ao longo da execução, e não apenas gerar um resumo no final. Essa decisão evita perder informações caso o notebook seja interrompido antes da criação do manifesto final.

A estrutura esperada de logs é:

```text
logs/training/04_run_training_events.jsonl
logs/training/04_run_training_summary.json

logs/training/runs/c2_struq_sft/seed_42/run_metadata.json
logs/training/runs/c2_struq_sft/seed_42/error.txt

logs/training/runs/c3_secalign_dpo/seed_42/run_metadata.json
logs/training/runs/c3_secalign_dpo/seed_42/error.txt

logs/training/runs/c4_ih_sft/seed_42/run_metadata.json
logs/training/runs/c4_ih_sft/seed_42/error.txt
```

O arquivo `04_run_training_events.jsonl` funciona como um histórico global de eventos. Cada linha deve registrar um evento independente, como início, conclusão ou falha de uma run.

Os diretórios em `logs/training/runs/` guardam informações específicas de cada combinação de `scenario_id` e seed. Isso permite identificar rapidamente qual run foi executada, onde o adaptador foi salvo, quais métricas de treino foram retornadas e qual erro ocorreu caso a execução tenha falhado.

A regra geral é:

```text
Uma run de treinamento, um diretório de adaptador e um diretório de log.
```

Assim, cada execução fica auditável e recuperável. Mesmo que o treinamento pare no meio, os logs das runs anteriores continuam disponíveis.


In [13]:
RUNS_LOG_DIR = LOG_DIR / "runs"
EVENTS_LOG_PATH = LOG_DIR / "04_run_training_events.jsonl"
SUMMARY_LOG_PATH = LOG_DIR / "04_run_training_summary.json"

RUNS_LOG_DIR.mkdir(parents=True, exist_ok=True)

logging_rows = []

for scenario_id, info in training_files.items():
    for seed in EXPERIMENT_SEEDS:
        run_log_dir = RUNS_LOG_DIR / scenario_id / f"seed_{seed}"
        run_log_dir.mkdir(parents=True, exist_ok=True)

        logging_rows.append({
            "scenario_id": scenario_id,
            "scenario_label": info["scenario_label"],
            "name": info["name"],
            "method": info["method"],
            "seed": seed,
            "run_log_dir": str(run_log_dir),
            "run_metadata": str(run_log_dir / "run_metadata.json"),
            "error_file": str(run_log_dir / "error.txt"),
            "global_events_log": str(EVENTS_LOG_PATH),
            "global_summary_log": str(SUMMARY_LOG_PATH),
        })

logging_df = pd.DataFrame(logging_rows)

logging_plan = {
    "log_dir": str(LOG_DIR),
    "runs_log_dir": str(RUNS_LOG_DIR),
    "global_events_log": str(EVENTS_LOG_PATH),
    "global_summary_log": str(SUMMARY_LOG_PATH),
    "scenario_id_policy": {
        "uses_canonical_scenario_ids": True,
        "canonical_scenario_ids": TRAINED_SCENARIOS,
        "matches_04_run_training": True,
    },
    "event_types": [
        "run_started",
        "run_completed",
        "run_failed",
    ],
    "per_run_files": {
        "metadata": "run_metadata.json",
        "error": "error.txt",
    },
    "logging_policy": {
        "incremental_events": True,
        "one_log_directory_per_scenario_seed": True,
        "record_traceback_on_failure": True,
        "write_summary_after_all_runs": True,
    },
    "runs": {
        f"{row['scenario_id']}_seed_{row['seed']}": {
            "scenario_id": row["scenario_id"],
            "scenario_label": row["scenario_label"],
            "name": row["name"],
            "method": row["method"],
            "seed": int(row["seed"]),
            "run_log_dir": row["run_log_dir"],
            "run_metadata": row["run_metadata"],
            "error_file": row["error_file"],
        }
        for _, row in logging_df.iterrows()
    },
}

display(logging_df)


,scenario_id,scenario_label,name,method,seed,run_log_dir,run_metadata,error_file,global_events_log,global_summary_log
0,c2_struq_sft,C2,StruQ-like SFT,sft,42,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
1,c2_struq_sft,C2,StruQ-like SFT,sft,123,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
2,c2_struq_sft,C2,StruQ-like SFT,sft,2026,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
3,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,42,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
4,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,123,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
5,c3_secalign_dpo,C3,SecAlign-like DPO,dpo,2026,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
6,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,42,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
7,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,123,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...
8,c4_ih_sft,C4,Instruction-Hierarchy-like SFT,sft,2026,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/runs/c...,/workspace/pi-defense-exp/logs/training/04_run...,/workspace/pi-defense-exp/logs/training/04_run...


## 16. Arquivos comuns de avaliação

Embora este notebook seja focado no treinamento, também é importante registrar quais arquivos serão usados na avaliação final.

Todos os cenários serão avaliados nos mesmos arquivos comuns:

```text
data/views/evaluation/test_clean.jsonl
data/views/evaluation/test_attacked_seen.jsonl
data/views/evaluation/test_attacked_unseen.jsonl
```

O arquivo `test_clean.jsonl` será usado para medir utilidade em entradas benignas.

Os arquivos `test_attacked_seen.jsonl` e `test_attacked_unseen.jsonl` serão usados para medir robustez contra ataques vistos e não vistos/adaptativos.

Essa separação garante que o treinamento e a avaliação não se misturem. Os arquivos de teste não devem ser usados para ajustar os modelos.

In [14]:
evaluation_files = {
    "test_clean": VIEWS_DIR / "evaluation" / "test_clean.jsonl",
    "test_attacked_seen": VIEWS_DIR / "evaluation" / "test_attacked_seen.jsonl",
    "test_attacked_unseen": VIEWS_DIR / "evaluation" / "test_attacked_unseen.jsonl",
}

evaluation_rows = []

for name, path in evaluation_files.items():
    evaluation_rows.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "line_count": count_jsonl_lines(path) if path.exists() else None,
    })

evaluation_files_df = pd.DataFrame(evaluation_rows)
display(evaluation_files_df)

missing_eval_files = evaluation_files_df.loc[~evaluation_files_df["exists"], "path"].tolist()
if missing_eval_files:
    raise FileNotFoundError("Arquivos de avaliação ausentes: " + ", ".join(missing_eval_files))

,name,path,exists,line_count
0,test_clean,/workspace/pi-defense-exp/data/views/evaluatio...,True,1876
1,test_attacked_seen,/workspace/pi-defense-exp/data/views/evaluatio...,True,9380
2,test_attacked_unseen,/workspace/pi-defense-exp/data/views/evaluatio...,True,5628


## 17. Criar `training_plan.yaml`

Nesta etapa, será criado o arquivo `configs/training_plan.yaml`.

Esse arquivo centraliza as decisões de treinamento que serão reutilizadas pelos próximos notebooks ou scripts.

Ele inclui:

```text
- modelo base;
- seed usada na criação do dataset;
- seeds experimentais usadas nas réplicas de treinamento;
- cenários que exigem treinamento;
- scenario_id canônico de cada cenário treinável;
- arquivos de treino e validação;
- diretórios-base de saída dos adaptadores;
- diretórios de saída por cenário e seed;
- plano de logs incrementais por scenario_id e seed;
- configuração LoRA/QLoRA;
- hiperparâmetros de SFT;
- hiperparâmetros de DPO;
- arquivos comuns de avaliação.
```

O objetivo é evitar que cada notebook de treinamento redefina manualmente os mesmos caminhos e parâmetros. Em vez disso, os próximos notebooks poderão carregar `training_plan.yaml` e executar o cenário correspondente.

Como C2, C3 e C4 serão treinados com as seeds `42`, `123` e `2026`, o plano de treinamento também registra quantas réplicas devem ser executadas, onde cada adaptador deve ser salvo e onde os logs de cada run devem ficar.

Para manter consistência com o notebook `04_run_training.ipynb`, os diretórios de logs usam os IDs canônicos:

```text
logs/training/runs/c2_struq_sft/seed_<seed>/
logs/training/runs/c3_secalign_dpo/seed_<seed>/
logs/training/runs/c4_ih_sft/seed_<seed>/
```

Além dos diretórios de adaptadores, o plano registra os caminhos esperados para logs globais e logs específicos por run. Essa informação será usada pelo notebook `04_run_training.ipynb` para registrar eventos incrementais de início, conclusão e falha de cada execução.


In [15]:
adapter_outputs_by_scenario = {}

for scenario_id, info in training_files.items():
    adapter_outputs_by_scenario[scenario_id] = {
        f"seed_{seed}": str(info["adapter_output"] / f"seed_{seed}")
        for seed in EXPERIMENT_SEEDS
    }

training_plan = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "base_model": model_plan,
    "seeds": seed_plan,
    "scenarios": {
        scenario_id: {
            "scenario_id": scenario_id,
            "scenario_label": info["scenario_label"],
            "name": info["name"],
            "method": info["method"],
            "train_file": str(info["train"]),
            "validation_file": str(info["validation"]),
            "adapter_output_base": str(info["adapter_output"]),
            "adapter_outputs_by_seed": adapter_outputs_by_scenario[scenario_id],
        }
        for scenario_id, info in training_files.items()
    },
    "logging": logging_plan,
    "lora_config": lora_config,
    "sft_training_config": sft_training_config,
    "dpo_training_config": dpo_training_config,
    "evaluation_files": {
        name: str(path)
        for name, path in evaluation_files.items()
    },
    "notes": [
        "C0 and C1 do not require training.",
        "C2 and C4 use SFT with clean + attacked_seen examples.",
        "C3 uses DPO with attacked_seen examples only.",
        "All scenarios must use the same base model.",
        "All scenarios must be evaluated on the same test files.",
        "c2_struq_sft, c3_secalign_dpo, and c4_ih_sft must each be trained once for each seed in EXPERIMENT_SEEDS.",
        "Each trained adapter must be saved under adapters/<defense_family>/seed_<seed>/.",
        "Training runs must write incremental logs under logs/training/ and logs/training/runs/<scenario_id>/seed_<seed>/.",
        "The scenario_id values are aligned with 04_run_training.ipynb.",
    ],
}

with open(TRAINING_PLAN_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(training_plan, f, sort_keys=False, allow_unicode=True)

print("Training plan salvo em:", TRAINING_PLAN_PATH)


Training plan salvo em: /workspace/pi-defense-exp/configs/training_plan.yaml


## 18. Checklist antes de iniciar o treinamento

Antes de executar os notebooks ou scripts de treinamento, é importante revisar uma lista de verificação.

O treinamento só deve começar se:

```text
- o kernel correto estiver selecionado;
- o manifesto do dataset existir;
- as views de treino e validação existirem;
- as contagens das views estiverem corretas;
- o modelo base tiver sido escolhido;
- os diretórios de adaptadores existirem;
- o arquivo training_plan.yaml tiver sido gerado;
- houver espaço em disco suficiente;
- a GPU estiver disponível;
- o token do Hugging Face estiver configurado, se o modelo exigir autenticação.
```

Também é recomendável executar primeiro um treinamento piloto com poucos exemplos ou poucos passos. Isso ajuda a identificar problemas de memória, tokenização, formato dos exemplos ou nomes de módulos LoRA antes de iniciar o treinamento completo.

Checklist metodológico:

```text
- C2, C3 e C4 partem do mesmo modelo base?
- C2 e C4 usam SFT?
- C3 usa DPO?
- C0 e C1 permanecem sem treinamento?
- Os arquivos de teste ficam fora do treinamento?
- O notebook registra os caminhos e configurações usados?
```

In [16]:
checklist = {
    "current_python_matches_expected": CURRENT_PYTHON == EXPECTED_PYTHON,
    "dataset_manifest_exists": DATASET_MANIFEST_PATH.exists(),
    "training_plan_exists": TRAINING_PLAN_PATH.exists(),
    "all_training_files_exist": bool(training_files_df["exists"].all()),
    "all_evaluation_files_exist": bool(evaluation_files_df["exists"].all()),
    "adapter_dirs_exist": bool(adapter_dirs_df["exists"].all()),
    "experiment_seeds_defined": EXPERIMENT_SEEDS == [42, 123, 2026],
    "scenario_ids_match_training_notebook": TRAINED_SCENARIOS == [
        "c2_struq_sft",
        "c3_secalign_dpo",
        "c4_ih_sft",
    ],
    "total_adapter_runs_expected": TOTAL_ADAPTER_RUNS == 9,
}

checklist_df = pd.DataFrame(
    [{"check": key, "status": value} for key, value in checklist.items()]
)

display(checklist_df)

if not all(checklist.values()):
    failed = [key for key, value in checklist.items() if not value]
    raise RuntimeError("Checklist incompleto: " + ", ".join(failed))

print("Checklist concluído com sucesso.")


,check,status
0,current_python_matches_expected,True
1,dataset_manifest_exists,True
2,training_plan_exists,True
3,all_training_files_exist,True
4,all_evaluation_files_exist,True
5,adapter_dirs_exist,True
6,experiment_seeds_defined,True
7,scenario_ids_match_training_notebook,True
8,total_adapter_runs_expected,True


Checklist concluído com sucesso.


## 19. Riscos e limitações documentadas

Esta etapa registra limitações importantes do plano de treinamento.

Primeiro, os cenários são aproximações experimentais. O objetivo é comparar princípios de defesa sob uma base comum, e não reproduzir integralmente os pipelines originais de StruQ, SecAlign ou Instruction Hierarchy.

Segundo, o treinamento será feito com adaptadores e com uma amostra controlada de dados. Isso torna o experimento viável para o contexto da disciplina, mas limita a generalização dos resultados.

Terceiro, o cenário SecAlign-like DPO usa pares sintéticos `chosen/rejected` derivados do próprio dataset atacado. Isso preserva a lógica de preferência, mas não equivale necessariamente ao dataset ou pipeline original do SecAlign.

Quarto, o uso de uma única GPU impõe restrições de memória. Por isso, a configuração inicial prioriza LoRA/QLoRA, batch pequeno e poucos epochs.

Quinto, os ataques `gcg` e `gcg_adaptive` foram definidos como templates GCG-like no notebook de dataset. Portanto, os resultados relacionados a esses ataques devem ser descritos como aproximações textuais, não como avaliação contra sufixos GCG otimizados por busca.

Essas limitações devem aparecer tanto nos manifestos quanto na discussão final do trabalho.

## 20. Gerar manifesto da documentação do treinamento

Nesta etapa final, será gerado o manifesto do notebook `03_training_documentation.ipynb`.

O manifesto registra:

```text
- data e hora de geração;
- caminho do projeto;
- Python usado;
- modelo base escolhido;
- arquivos de treino e validação;
- arquivos comuns de avaliação;
- configurações LoRA/QLoRA;
- configurações SFT;
- configurações DPO;
- diretórios de saída dos adaptadores;
- plano de logs incrementais;
- checklist de pré-treinamento;
- caminhos dos manifestos anteriores.
```

Serão criados dois arquivos:

```text
manifests/training/03_training_documentation_manifest.json
manifests/training/03_training_documentation_manifest.md
```

O arquivo JSON é útil para leitura por scripts. O arquivo Markdown é útil para inspeção manual e documentação da reprodução.

O manifesto também registra a política de logs planejada para o notebook `04_run_training.ipynb`. Isso garante que a estratégia de auditoria das runs fique documentada antes da execução do treinamento.


In [17]:
manifest_json_path = MANIFEST_DIR / "03_training_documentation_manifest.json"
manifest_md_path = MANIFEST_DIR / "03_training_documentation_manifest.md"

training_file_summary = {
    f"{row['scenario_id']}_{row['split']}": {
        "scenario_id": row["scenario_id"],
        "scenario_label": row["scenario_label"],
        "name": row["name"],
        "method": row["method"],
        "split": row["split"],
        "path": row["path"],
        "exists": bool(row["exists"]),
        "line_count": int(row["line_count"]) if pd.notna(row["line_count"]) else None,
    }
    for _, row in training_files_df.iterrows()
}

evaluation_file_summary = {
    row["name"]: {
        "path": row["path"],
        "exists": bool(row["exists"]),
        "line_count": int(row["line_count"]) if pd.notna(row["line_count"]) else None,
    }
    for _, row in evaluation_files_df.iterrows()
}

adapter_dir_summary = {
    f"{row['scenario_id']}_seed_{row['seed']}": {
        "scenario_id": row["scenario_id"],
        "scenario_label": row["scenario_label"],
        "name": row["name"],
        "method": row["method"],
        "seed": int(row["seed"]),
        "adapter_output_base": row["adapter_output_base"],
        "adapter_output": row["adapter_output"],
        "exists": bool(row["exists"]),
    }
    for _, row in adapter_dirs_df.iterrows()
}

manifest = {
    "notebook": "03_training_documentation",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "expected_kernel": EXPECTED_KERNEL,
    "expected_python": str(EXPECTED_PYTHON),
    "current_python": str(CURRENT_PYTHON),
    "python": {
        "version": sys.version,
        "executable": str(CURRENT_PYTHON),
    },
    "system": {
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
    },
    "previous_manifests": {
        "environment": str(ENVIRONMENT_MANIFEST_PATH),
        "environment_exists": ENVIRONMENT_MANIFEST_PATH.exists(),
        "dataset": str(DATASET_MANIFEST_PATH),
        "dataset_exists": DATASET_MANIFEST_PATH.exists(),
    },
    "experiment_config": str(EXPERIMENT_CONFIG_PATH),
    "training_plan": str(TRAINING_PLAN_PATH),
    "base_model": model_plan,
    "seeds": seed_plan,
    "scenarios": scenario_rows,
    "training_files": training_file_summary,
    "evaluation_files": evaluation_file_summary,
    "adapter_directories": adapter_dir_summary,
    "logging": logging_plan,
    "lora_config": lora_config,
    "sft_training_config": sft_training_config,
    "dpo_training_config": dpo_training_config,
    "checklist": checklist,
    "limitations": [
        "The training scenarios are controlled approximations of StruQ, SecAlign, and Instruction Hierarchy.",
        "C2 and C4 use SFT; C3 uses DPO.",
        "C0 and C1 do not train adapters.",
        "All scenarios must use the same base model.",
        "All scenarios must be evaluated on the same evaluation files.",
        "c2_struq_sft, c3_secalign_dpo, and c4_ih_sft must be trained once for each experimental seed.",
        "The dataset seed is fixed separately from the training replicate seeds.",
        "Training logs must be written incrementally and separated by scenario_id and seed.",
        "GCG and GCG-adaptive attacks are GCG-like templates in this version, not optimized adversarial suffixes.",
        "A single-GPU RunPod environment may require reducing sequence length, batch size, or model size.",
    ],
}

with open(manifest_json_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)


def make_table_md(rows: list[dict], columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    lines = [header, sep]

    for row in rows:
        lines.append("| " + " | ".join(str(row.get(col, "")) for col in columns) + " |")

    return "\n".join(lines)


training_files_md_rows = [
    {
        "Scenario ID": row["scenario_id"],
        "Label": row["scenario_label"],
        "Method": row["method"],
        "Split": row["split"],
        "Lines": row["line_count"],
        "Path": f"`{row['path']}`",
    }
    for _, row in training_files_df.iterrows()
]

evaluation_files_md_rows = [
    {
        "Name": row["name"],
        "Lines": row["line_count"],
        "Path": f"`{row['path']}`",
    }
    for _, row in evaluation_files_df.iterrows()
]

adapter_dirs_md_rows = [
    {
        "Scenario ID": row["scenario_id"],
        "Label": row["scenario_label"],
        "Method": row["method"],
        "Seed": row["seed"],
        "Path": f"`{row['adapter_output']}`",
    }
    for _, row in adapter_dirs_df.iterrows()
]

logging_md_rows = [
    {
        "Scenario ID": row["scenario_id"],
        "Label": row["scenario_label"],
        "Method": row["method"],
        "Seed": row["seed"],
        "Run log dir": f"`{row['run_log_dir']}`",
        "Metadata": f"`{row['run_metadata']}`",
        "Error file": f"`{row['error_file']}`",
    }
    for _, row in logging_df.iterrows()
]

training_files_table_md = make_table_md(
    training_files_md_rows,
    ["Scenario ID", "Label", "Method", "Split", "Lines", "Path"],
)

evaluation_files_table_md = make_table_md(
    evaluation_files_md_rows,
    ["Name", "Lines", "Path"],
)

adapter_dirs_table_md = make_table_md(
    adapter_dirs_md_rows,
    ["Scenario ID", "Label", "Method", "Seed", "Path"],
)

logging_table_md = make_table_md(
    logging_md_rows,
    ["Scenario ID", "Label", "Method", "Seed", "Run log dir", "Metadata", "Error file"],
)

checklist_table_md = make_table_md(
    [{"Check": key, "Status": value} for key, value in checklist.items()],
    ["Check", "Status"],
)

experiment_seeds_md = ", ".join([f"`{seed}`" for seed in EXPERIMENT_SEEDS])
trained_scenarios_md = ", ".join([f"`{scenario_id}`" for scenario_id in TRAINED_SCENARIOS])

manifest_md = f"""# Manifesto de documentação do treinamento — Notebook 03

## Identificação

- Notebook: `03_training_documentation`
- Gerado em UTC: `{manifest["created_at_utc"]}`
- Diretório raiz do projeto: `{PROJECT_ROOT}`
- Kernel esperado: `{EXPECTED_KERNEL}`
- Python esperado: `{EXPECTED_PYTHON}`
- Python atual: `{CURRENT_PYTHON}`

## Manifestos anteriores

- Ambiente: `{ENVIRONMENT_MANIFEST_PATH}` — existe: `{ENVIRONMENT_MANIFEST_PATH.exists()}`
- Dataset: `{DATASET_MANIFEST_PATH}` — existe: `{DATASET_MANIFEST_PATH.exists()}`

## Modelo base

- Modelo base: `{BASE_MODEL_ID}`
- Mesmo modelo base para todos os cenários: `{model_plan["same_base_model_for_all_scenarios"]}`

## Seeds experimentais

- Seed do dataset: `{DATASET_SEED}`
- Seeds de treinamento: {experiment_seeds_md}
- Scenario IDs treináveis: {trained_scenarios_md}
- Réplicas por cenário treinável: `{REPLICATES_PER_TRAINED_SCENARIO}`
- Total de execuções de treinamento de adaptadores: `{TOTAL_ADAPTER_RUNS}`

A seed do dataset controla a criação da base experimental e deve permanecer fixa. As seeds de treinamento controlam as réplicas dos cenários `c2_struq_sft`, `c3_secalign_dpo` e `c4_ih_sft`, permitindo calcular médias, desvios padrão e intervalos de confiança na avaliação final.

## Cenários com treinamento

{training_files_table_md}

## Arquivos comuns de avaliação

{evaluation_files_table_md}

## Diretórios de adaptadores por seed

{adapter_dirs_table_md}

## Plano de logs incrementais

- Log global de eventos: `{EVENTS_LOG_PATH}`
- Resumo global de treinamento: `{SUMMARY_LOG_PATH}`
- Diretório-base de logs por run: `{RUNS_LOG_DIR}`

Cada run deve registrar eventos incrementais de início, conclusão e falha. Em caso de erro, o traceback deve ser salvo em `error.txt` dentro do diretório específico da combinação `scenario_id`/seed.

{logging_table_md}

## Configuração LoRA/QLoRA

```yaml
{yaml.safe_dump(lora_config, sort_keys=False, allow_unicode=True)}
```

## Configuração SFT

```yaml
{yaml.safe_dump(sft_training_config, sort_keys=False, allow_unicode=True)}
```

## Configuração DPO

```yaml
{yaml.safe_dump(dpo_training_config, sort_keys=False, allow_unicode=True)}
```

## Checklist

{checklist_table_md}

## Observações e limitações

- Os cenários de treinamento são aproximações controladas de StruQ, SecAlign e Instruction Hierarchy.
- C2 e C4 usam SFT.
- C3 usa DPO.
- C0 e C1 não treinam adaptadores.
- Todos os cenários devem partir do mesmo modelo base.
- Todos os cenários devem ser avaliados nos mesmos arquivos comuns de avaliação.
- `c2_struq_sft`, `c3_secalign_dpo` e `c4_ih_sft` devem ser treinados uma vez para cada seed experimental.
- Os logs de treinamento devem ser incrementais e separados por `scenario_id`/seed.
- A seed do dataset é fixa e separada das seeds das réplicas de treinamento.
- Ataques `gcg` e `gcg_adaptive` são templates GCG-like nesta versão, não sufixos adversariais otimizados.
- O ambiente RunPod com uma GPU pode exigir redução de sequência, batch, quantidade de exemplos ou tamanho do modelo.
"""

with open(manifest_md_path, "w", encoding="utf-8") as f:
    f.write(manifest_md)

print("Manifesto JSON criado em:", manifest_json_path)
print("Manifesto Markdown criado em:", manifest_md_path)


Manifesto JSON criado em: /workspace/pi-defense-exp/manifests/training/03_training_documentation_manifest.json
Manifesto Markdown criado em: /workspace/pi-defense-exp/manifests/training/03_training_documentation_manifest.md
